**Simplified BERT for Masked Language Modeling (MLM) using PyTorch**
In this project, I implemented a BERT-style encoder model from scratch using PyTorch, focusing solely on the Masked Language Modeling (MLM) objective. The model architecture is a simplified version of BERT with the following key modifications:

Only MLM objective is used — the Next Sentence Prediction (NSP) task is excluded to streamline training

The model does not use the [CLS] token, as it's not required for MLM

The model learns to predict masked tokens within input sequences, encouraging contextual understanding

Key features include:

A custom-built Transformer Encoder Layer, reused from previous assignments, with Multi-Head Attention, Feed Forward Networks, and Layer Normalization

Embedding layers for token and position embeddings

ReLU activation and Stochastic Gradient Descent (SGD) optimizer for training

The model is trained on a small corpus and designed to overfit on it to validate correctness

For expanded experimentation, larger datasets and Hugging Face’s tokenizer library can be used for efficient tokenization and masking. This project provides a foundational understanding of how BERT learns language representations purely from masked token prediction.

# Installations

In [ ]:
!pip install torchdata==0.6.0 # to be compatible with torch 2.0
!pip install portalocker==2.0.0
!pip install -U torchtext==0.15.1

# Common Imports

In [ ]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

#text lib
import torchtext

# tokenizer
from torchtext.data.utils import get_tokenizer

#build vocabulary
from torchtext.vocab import vocab
from torchtext.vocab import build_vocab_from_iterator

# get input_ids (numericalization)
from torchtext.transforms import VocabTransform

# get embeddings
from torch.nn import Embedding

from  pprint import pprint
from yaml import safe_load
import copy
import numpy as np
import math

In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Tokenize the given text

In [ ]:
batch_size = 10

In [ ]:
class Tokenizer(object):

  def __init__(self,text):
    self.text = text
    self.word_tokenizer = get_tokenizer(tokenizer="basic_english",language='en')
    self.vocab_size = None

  def get_tokens(self):
    for sentence in self.text.strip().split('\n'):
      yield self.word_tokenizer(sentence)

  def build_vocab(self):
    v = build_vocab_from_iterator(self.get_tokens(),
                                  min_freq=1,specials=['<unk>','<mask>'])
    v.set_default_index(v['<unk>']) # index of OOV
    self.vocab_size = len(v)
    return v

  def token_ids(self):
    v = self.build_vocab()
    vt = VocabTransform(v)
    num_tokens = len(self.word_tokenizer(self.text))
    max_seq_len = np.ceil(num_tokens/batch_size)
    data = torch.zeros(size=(1,num_tokens))
    data = vt(self.word_tokenizer(self.text))
    data = torch.tensor(data,dtype=torch.int64)
    return data.reshape(batch_size,torch.tensor(max_seq_len,dtype=torch.int64))



In [ ]:
text = """Best known for the invention of Error Correcting Codes, he was a true polymath who applied his mathematical and problem-solving skills to numerous disciplines.
Reflecting on the significant benefits I received from Hamming, I decided to develop a tribute to his legacy. There has not been a previous biography of Hamming, and the few articles about him restate known facts and assumptions and leave us with open questions.
One thought drove me as I developed this legacy project: An individual's legacy is more than a list of their attempts and accomplishments. Their tribute should also reveal the succeeding generations they inspired and enabled and what each attempted and achieved.
This book is a unique genre containing my version of a biography that intertwines the story "of a life" and a multi-player memoir with particular events and turning points recalled by those, including me, who he inspired and enabled.
Five years of research uncovered the people, places, opportunities, events, and influences that shaped Hamming. I discovered unpublished information, stories, photographs, videos, and personal remembrances to chronicle his life, which helped me put Hamming's
legacy in the context I wanted.The result demonstrates many exceptional qualities, including his noble pursuit of excellence and helping others. Hamming paid attention to the details, his writings continue to influence, and his guidance is a timeless gift to the world.
This biography is part of """

In [ ]:
Tk = Tokenizer(text)

In [ ]:
input_ids = Tk.token_ids()
print(input_ids.shape)

torch.Size([10, 26])


* We need to mask some words randomly based on the mask probability
* The token id for the [mask] is 1
* The function given below takes in the input ids and replaces some of the ids by 1 (token id for the [mask])
* Since the loss is computed only over the predictions of masked tokens, we replace all non-masked input ids by -100

In [ ]:
def getdata(ip_ids,mask_token_id,mask_prob=0.2):
  masked_ids = copy.deepcopy(ip_ids)
  mask_random_idx = torch.randn_like(ip_ids,dtype=torch.float64)>(1-mask_prob)
  masked_ids[mask_random_idx]=mask_token_id
  labels = copy.deepcopy(ip_ids)
  neg_mask = ~mask_random_idx
  labels[neg_mask]=torch.tensor(-100)
  return (masked_ids,labels,mask_random_idx)

In [ ]:
mask_token_id = torch.tensor([1],dtype=torch.int64)
x,y,mask_mtx = getdata(input_ids,mask_token_id)
print(x[0,:],'\n',y[0,:])

tensor([  1,  23,  69,   5,  85,   7,  63,  53,  49,   2,  20, 148,   1, 139,
        110,   1,  36,   9,  89,   3, 112,   1,   8,   1,  59,   1]) 
 tensor([  45, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
           6, -100, -100,   29, -100, -100, -100, -100, -100,  129, -100,   96,
        -100,    4])


* Now we have our inputs and labels stored in x and y,respectively
* It is always good to test the implementation by displaying the input sentence with masked tokens

In [ ]:
v = Tk.build_vocab()
words = []
for idx in x[0,:]:
  words.append(v.vocab.get_itos()[idx.item()])
print(' '.join(words))

<mask> known for the invention of error correcting codes , he was <mask> true polymath <mask> applied his mathematical and problem-solving <mask> to <mask> disciplines <mask>


* Also display the words that are masked

In [ ]:
words = []
for idx in y[0,:]:
  if idx != -100:
    words.append(v.vocab.get_itos()[idx.item()])
print(' '.join(words))

best a who skills numerous .


# Configuration

In [ ]:
vocab_size = Tk.vocab_size
seq_len = x.shape[1]
embed_dim = 32
dmodel = embed_dim
dq = torch.tensor(4)
dk = torch.tensor(4)
dv = torch.tensor(4)
heads = torch.tensor(8)
d_ff = 4*dmodel

# Model

In [ ]:
class MHA(nn.Module):
  def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHA,self).__init__()
    self.num_heads = heads
    self.d_model = dmodel
    self.dq = dq
    self.dk = dk
    self.dv = dv

    # Linear layers for queries, keys, and values
    # torch.manual_seed(43)
    self.W_q = nn.Linear(dmodel, dq * heads)
    # torch.manual_seed(44)
    self.W_k = nn.Linear(dmodel, dk * heads)
    # torch.manual_seed(45)
    self.W_v = nn.Linear(dmodel, dv * heads)
    # Linear layer for output
    # torch.manual_seed(46)
    self.W_o = nn.Linear(heads * dv, dmodel)

  def forward(self,H=None):
    '''
    Input: Size [BSxTxdmodel]
    Output: Size[BSxTxdmodel]
    '''
    batch_size, seq_len, dmodel = H.size()

    # Linear transformations
    Q = self.W_q(H).view(batch_size, seq_len, self.num_heads, self.dq).transpose(1, 2)  # [BS x heads x T x dq]
    K = self.W_k(H).view(batch_size, seq_len, self.num_heads, self.dk).transpose(1, 2)  # [BS x heads x T x dk]
    V = self.W_v(H).view(batch_size, seq_len, self.num_heads, self.dv).transpose(1, 2)  # [BS x heads x T x dv]

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.dk ** 0.5)  # [BS x heads x T x T]
    attn_weights = F.softmax(scores, dim=-1)  # [BS x heads x T x T]
    output = torch.matmul(attn_weights, V)  # [BS x heads x T x dv]

    # Concatenate heads and apply output linear transformation
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.num_heads * self.dv)  # [BS x T x (heads * dv)]
    out = self.W_o(output)  # [BS x T x d_model]

    return out


class FFN(nn.Module):
  def __init__(self,dmodel,d_ff,layer=0):
    super(FFN,self).__init__()
    # First linear layer to expand the dimension
    # torch.manual_seed(47)
    self.fc1 = nn.Linear(dmodel, d_ff)
    # Second linear layer to project it back to the original dimension
    # torch.manual_seed(48)
    self.fc2 = nn.Linear(d_ff, dmodel)
    # ReLU activation
    self.relu = nn.ReLU()

  def forward(self,x):
    '''
    input: size [BSxTxdmodel]
    output: size [BSxTxdmodel]
    '''
    return self.fc2(self.relu(self.fc1(x)))



class Prediction(nn.Module):
  def __init__(self,dmodel,vocab_size):
    super(Prediction,self).__init__()
    torch.manual_seed(49)
    self.linear = nn.Linear(dmodel, vocab_size)

  def forward(self,representations):
    '''
    input: size [bsxTxdmodel]
    output: size [bsxTxvocab_size]
    Note: Do not apply the softmax. Just return the output of linear transformation
    '''
    out = self.linear(representations)
    return out


class PositionalEncoding(nn.Module):
  def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create a matrix of (max_len, d_model) with positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))  # (d_model/2,)

        # Compute the sine and cosine for each position and dimension
        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sine to even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Apply cosine to odd indices

        pe = pe.unsqueeze(0)  # Add a batch dimension (1, max_len, d_model)
        self.register_buffer('pe', pe)  # Register pe as a persistent buffer

  def forward(self, x):
        # x=x.view(-1,x.shape[0],x.shape[1])
        x = x + self.pe[:, :x.size(1)]  # x.size(1) is the length of the sequence
        return self.dropout(x)

class Embed(nn.Module):
  def __init__(self,vocab_size,embed_dim):
      super(Embed,self).__init__()
      torch.manual_seed(70)
      self.embed = nn.Embedding(vocab_size,embed_dim) # seed 70
      self.pe = PositionalEncoding(embed_dim)

  def forward(self,x):
      out = self.pe(self.embed(x))
      return out


class EncoderLayer(nn.Module):

    def __init__(self,dmodel,dq,dk,dv,d_ff,heads):
      super(EncoderLayer,self).__init__()
      self.mha = MHA(dmodel,dq,dk,dv,heads)
      self.layer_norm_1 = torch.nn.LayerNorm(dmodel)
      self.layer_norm_2 = torch.nn.LayerNorm(dmodel)
      self.ffn = FFN(dmodel,d_ff)

    def forward(self,x):
      mhaoutput=self.mha(x)
      mhaoutput=mhaoutput + x
      layernorm1output=self.layer_norm_1(mhaoutput)
      ffnoutput=self.ffn(layernorm1output)
      ffnoutput=ffnoutput + layernorm1output
      out=self.layer_norm_2(ffnoutput)

      return out

In [ ]:
class BERT(nn.Module):

  def __init__(self,vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=1):
    super(BERT,self).__init__()
    self.embed_lookup = Embed(vocab_size,embed_dim)
    self.enc_layers = nn.ModuleList(copy.deepcopy(EncoderLayer(dmodel,dq,dk,dv,d_ff,heads)) for i in range(num_layers))
    self.predict = Prediction(dmodel,vocab_size)

  def forward(self,input_ids):
    x = self.embed_lookup(input_ids)
    for enc_layer in self.enc_layers:
      x = enc_layer(x)
    out = self.predict(x)
    return out

In [ ]:
model = BERT(vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=1)
optimizer = optim.SGD(model.parameters(),lr=0.01)
criterion = nn.CrossEntropyLoss()

# Training the model

In [ ]:
def train(token_ids,labels,epochs=1000):
  loss_trace = []
  for epoch in range(epochs):
    out = model(token_ids)
    loss = criterion(out.view(-1, out.size(-1)), labels.view(-1))
    loss_trace.append(loss.item())
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()


In [ ]:
train(x,y,20000)

Streaming output truncated to the last 5000 lines.
Epoch 15001/20000, Loss: 0.025151045992970467
Epoch 15002/20000, Loss: 0.04489528015255928
Epoch 15003/20000, Loss: 0.05022509768605232
Epoch 15004/20000, Loss: 0.02534763514995575
Epoch 15005/20000, Loss: 0.019755015149712563
Epoch 15006/20000, Loss: 0.010801024734973907
Epoch 15007/20000, Loss: 0.015016422607004642
Epoch 15008/20000, Loss: 0.02186415158212185
Epoch 15009/20000, Loss: 0.02729330025613308
Epoch 15010/20000, Loss: 0.07869325578212738
Epoch 15011/20000, Loss: 0.02999088540673256
Epoch 15012/20000, Loss: 0.010091080330312252
Epoch 15013/20000, Loss: 0.026865970343351364
Epoch 15014/20000, Loss: 0.0643572211265564
Epoch 15015/20000, Loss: 0.0663876086473465
Epoch 15016/20000, Loss: 0.0179576575756073
Epoch 15017/20000, Loss: 0.02108205482363701
Epoch 15018/20000, Loss: 0.02160164900124073
Epoch 15019/20000, Loss: 0.025450173765420914
Epoch 15020/20000, Loss: 0.03460191562771797
Epoch 15021/20000, Loss: 0.034506842494010925

In [ ]:
with torch.inference_mode():
  predictions = torch.argmax(model(x),dim=-1)

In [ ]:
v = Tk.build_vocab()
masked_words = []
predicted_words=[]
for index,idx in enumerate(y.flatten()):
  # to display only the masked tokens
  if idx != -100:
    masked_words.append(v.vocab.get_itos()[idx.item()])
    predicted_words.append(v.vocab.get_itos()[predictions.flatten()[index].item()])
print('Masked Words: ')
print(' '.join(masked_words))
print('Predicted Words: ')
print(' '.join(predicted_words))

Masked Words: 
best a who skills numerous . reflecting i develop his . not a previous , him questions thought me as legacy s their . should the they inspired what achieved version and with who research events discovered information stories and remembrances to chronicle life which context the many qualities excellence details , this biography is of
Predicted Words: 
best a who skills numerous . reflecting i develop his . not a previous , him questions thought me as legacy s their . should the they inspired what achieved version and with who research events discovered information stories and remembrances to chronicle life which context the many qualities excellence details , this biography is of
